# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhanish0711/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook documents **Assignment ML-10**: turning our validated machine learning ranking model into a structured **Content Action Playbook** with explicit reason codes, human review checklists, strict no-go automation boundaries, monitoring/retrain triggers, and exported artifacts for the final research paper.

## 1. Ranked actions + reason codes

### Action Taxonomy & Reason Codes
A raw model probability score $P(\text{decline} \mid X)$ is an operational signal, not a complete instruction. We map model scores and observable search signals into four distinct editorial action archetypes:

| Action Archetype | Trigger Criteria | Primary Reason Code | Editorial Action Playbook |
|---|---|---|---|
| **`comprehensive_content_refresh`** | Model Prob $\ge 0.65$ & `days_since_last_update >= 180` & `impressions_90d >= 500` | `stale_high_demand_decay` | Update outdated facts, expand thin sections with fresh industry developments, improve internal linking, refresh publishing timestamp. |
| **`title_meta_rewrite`** | `avg_position <= 10` & `ctr < 0.50%` & `impressions_90d >= 250` | `page_one_low_ctr` | Rewrite meta title for search intent match, craft persuasive meta description, test schema markup to improve SERP click capture. |
| **`engagement_ux_optimization`** | `sessions_90d >= 50` & (`engagement_rate < 30%` or `scroll_rate < 30%`) | `high_bounce_weak_scroll` | Improve content readability, add interactive table of contents, break up walls of text, improve mobile layout and load speed. |
| **`performance_monitoring`** | `impressions_90d >= 1000` & Model Prob $< 0.40$ | `stable_high_volume` | High-value evergreen content performing well; maintain monitoring and protect rankings against SERP shifts. |

In [1]:
import pandas as pd, numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from pathlib import Path
import json

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'].str.lower() == 'down').astype(int)

features = ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'engagement_rate', 'content_age_days', 'word_count']
X = df[features].fillna(0)
y = df['is_declining'].values
groups = df['client_id'].values

# Train Gradient Boosting Model on full client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X.iloc[train_idx], y[train_idx])
df['model_prob'] = gb.predict_proba(X)[:, 1]

# Assign Action Archetypes and Reason Codes
def assign_playbook_action(row):
    if row['model_prob'] >= 0.65 and row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return 'comprehensive_content_refresh', 'stale_high_demand_decay', 1
    elif row['avg_position'] > 0 and row['avg_position'] <= 10 and row['ctr'] < 0.50 and row['impressions_90d'] >= 250:
        return 'title_meta_rewrite', 'page_one_low_ctr', 2
    elif row['sessions_90d'] >= 50 and (row['engagement_rate'] < 30 or row['scroll_rate'] < 30):
        return 'engagement_ux_optimization', 'high_bounce_weak_scroll', 3
    elif row['impressions_90d'] >= 1000 and row['model_prob'] < 0.40:
        return 'performance_monitoring', 'stable_high_volume', 4
    else:
        return 'no_action_required', 'low_priority_content', 5

action_results = df.apply(assign_playbook_action, axis=1)
df['recommended_action'] = [r[0] for r in action_results]
df['reason_code'] = [r[1] for r in action_results]
df['priority_tier'] = [r[2] for r in action_results]

# Summary of Recommended Actions
action_summary = df['recommended_action'].value_counts().reset_index()
action_summary.columns = ['Recommended Action', 'Page Count']
action_summary['Percentage (%)'] = (action_summary['Page Count'] / len(df) * 100).round(2)
print('=== CONTENT ACTION PLAYBOOK DISTRIBUTION ===')
print(action_summary.to_string(index=False))


=== CONTENT ACTION PLAYBOOK DISTRIBUTION ===
           Recommended Action  Page Count  Percentage (%)
           no_action_required       19215           64.05
           title_meta_rewrite        6595           21.98
   engagement_ux_optimization        3501           11.67
       performance_monitoring         673            2.24
comprehensive_content_refresh          16            0.05


## 2. Intended use and limits

### Intended Operational Use
* **Primary Target Audience:** SEO Directors, Content Marketing Leads, and Editorial Strategists.
* **Workflow Integration:** Operates as a **weekly candidate queue generator**. Editorial teams review the top 20 to 50 flagged pages during weekly sprint planning, allocating writer hours to high-confidence opportunities.

### Operational & Analytical Limits
1. **Decision-Support, Not Autonomous Execution:** The model scores decay risk based on numerical search signals; it cannot evaluate whether writing quality, brand tone, or technical nuance is adequate.
2. **Minimum Data History Requirement:** Not valid for newly published articles (<90 days old) where impression and engagement baselines have not stabilized.
3. **Non-Causal Ranking:** A high opportunity score indicates statistical decay risk, not a mathematical guarantee that a refresh will restore top rankings.

In [2]:
# Declaration of Intended Use & Boundaries
playbook_boundaries = {
    'Intended Role': 'Weekly editorial review candidate prioritization',
    'Target Capacity': 'Top 20 to 50 articles per client per sprint',
    'Exclusion Limit 1': 'Content age < 90 days (insufficient baseline data)',
    'Exclusion Limit 2': 'Autonomous CMS publishing without human verification'
}
for k, v in playbook_boundaries.items():
    print(f'{k:20s}: {v}')


Intended Role       : Weekly editorial review candidate prioritization
Target Capacity     : Top 20 to 50 articles per client per sprint
Exclusion Limit 1   : Content age < 90 days (insufficient baseline data)
Exclusion Limit 2   : Autonomous CMS publishing without human verification


## 3. Human review + the no-go list

### Human Review Protocol (Pre-Action Checklist)
Before making changes to any flagged page, human editors must verify three criteria:
1. **SERP Layout Verification:** Check if Google introduced new SERP features (AI Overviews, Knowledge Graphs, Video carousels) that artificially depress CTR without ranking loss.
2. **Keyword Cannibalization Check:** Verify that traffic wasn't simply absorbed by another sibling article on the same domain.
3. **Search Intent Shift:** Verify whether user search intent evolved (e.g. from general overview to specific product comparison).

### The Strict No-Go List (Never Automate)
* ❌ **No Automated Deletions / 410 Pruning:** Never bulk-delete pages without editorial and backlink review.
* ❌ **No Unreviewed AI Rewrites:** Never publish AI-generated content refreshes without subject-matter expert review.
* ❌ **No Automated URL / Slug Changes:** Never modify live URLs or redirects automatically, as broken redirects cause catastrophic loss of historical ranking authority.

In [3]:
# Verification of No-Go Protocol
no_go_rules = [
    'No automated page deletions or bulk 410 HTTP pruning',
    'No unedited AI text generation published directly to CMS',
    'No programmatic URL slug alterations or mass redirect creation'
]
print('=== STRICT NO-GO AUTOMATION BOUNDARIES ===')
for idx, rule in enumerate(no_go_rules, 1):
    print(f'{idx}. {rule}')


=== STRICT NO-GO AUTOMATION BOUNDARIES ===
1. No automated page deletions or bulk 410 HTTP pruning
2. No unedited AI text generation published directly to CMS
3. No programmatic URL slug alterations or mass redirect creation


## 4. Monitoring / retrain triggers

To ensure the playbook remains reliable over time, we establish clear **model degradation and retrain triggers**:

1. **Performance Metric Drift (Precision@50 < 0.60):**
   * If the model's out-of-sample Precision@50 drops below 0.60 on trailing monthly audits, trigger a full model retraining and feature re-weighting.
2. **SERP Macro Shifts (Core Algorithm Updates):**
   * Major Google search core updates alter baseline CTR-by-position curves. Recalibrate position tier benchmarks after broad algorithm rollouts.
3. **Scheduled Recalibration Cadence:**
   * Re-train model weights every **60 days** using a rolling 12-month trailing window.

In [4]:
# Retrain Trigger Rules
monitoring_triggers = {
    'Precision@50 Threshold': 'Drop below 0.600 on holdout validation',
    'Algorithm Update Trigger': 'Re-estimate CTR curves post-core update',
    'Standard Retrain Cadence': 'Every 60 days on rolling historical facts'
}
print('=== PLAYBOOK MONITORING & RETRAIN TRIGGERS ===')
for trigger, condition in monitoring_triggers.items():
    print(f'{trigger:26s}: {condition}')


=== PLAYBOOK MONITORING & RETRAIN TRIGGERS ===
Precision@50 Threshold    : Drop below 0.600 on holdout validation
Algorithm Update Trigger  : Re-estimate CTR curves post-core update
Standard Retrain Cadence  : Every 60 days on rolling historical facts


## 5. Exports for the paper

Below we export the final ranked action queue CSV, summary metrics JSON, and portfolio distribution charts to `work/outputs/` and `work/figures/` for inclusion in the final research paper:

In [5]:
import matplotlib.pyplot as plt

# 1. Sort queue by priority and model probability
ranked_queue = df.sort_values(by=['priority_tier', 'model_prob', 'impressions_90d'], ascending=[True, False, False]).reset_index(drop=True)
queue_export_cols = [
    'content_id', 'client_id', 'priority_tier', 'recommended_action', 'reason_code', 
    'model_prob', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'is_declining'
]

queue_csv = Path('work/outputs/action_playbook_queue.csv')
ranked_queue[queue_export_cols].to_csv(queue_csv, index=False)
print(f'1. Wrote Action Queue CSV: {queue_csv} ({len(ranked_queue):,} rows)')

# 2. Export Playbook Metrics JSON
playbook_metrics = {
    'total_inventory_scored': int(len(ranked_queue)),
    'action_breakdown': df['recommended_action'].value_counts().to_dict(),
    'top50_precision_at_50': float(ranked_queue.head(50)['is_declining'].mean()),
    'top_recommended_action': str(df['recommended_action'].value_counts().index[0])
}
metrics_json = Path('work/outputs/playbook_summary.json')
with open(metrics_json, 'w') as f:
    json.dump(playbook_metrics, f, indent=2)
print(f'2. Wrote Playbook Summary JSON: {metrics_json}')

# 3. Export Action Distribution Chart
fig_dir = Path('work/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / 'action_distribution.png'

plt.figure(figsize=(10, 5))
counts = df['recommended_action'].value_counts()
plt.barh(counts.index, counts.values, color='#1f77b4', edgecolor='black')
plt.xlabel('Number of Pages')
plt.title('Content Action Playbook - Portfolio Action Distribution')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.close()
print(f'3. Wrote Action Distribution Figure: {fig_path}')


1. Wrote Action Queue CSV: work\outputs\action_playbook_queue.csv (30,000 rows)
2. Wrote Playbook Summary JSON: work\outputs\playbook_summary.json


3. Wrote Action Distribution Figure: work\figures\action_distribution.png


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.